# TP05: Estructuración y Agregación
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 3: Herramientas en la Nube de Modelado de Datos

---

### 🎯 Objetivos del Trabajo Práctico

1. Guardar dataframes como **tablas estructuradas**
2. Aplicar **funciones de agregación** sobre métricas comerciales
3. Crear **vistas** y esquemas lógicos
4. Realizar **agrupamientos complejos**
5. Transformar grupos con funciones personalizadas

---

### 📁 Caso de Estudio: Tablas de Ventas Agregadas

Crearemos tablas estructuradas con métricas agregadas de la panadería.

### 🕰️ Duración Estimada: 2 horas

In [0]:
# Importar librerías
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Inicializar Spark
spark = SparkSession.builder.getOrCreate()

print("✅ Librerías importadas")
print(f"Spark version: {spark.version}")

## Parte 1: Carga de Datos como Spark DataFrames

### 📂 De Pandas a Spark

Vamos a cargar los CSV con Pandas y convertirlos a Spark DataFrames para trabajar con tablas estructuradas.

In [0]:
# Cargar datos con Pandas
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_productos_pd = pd.read_csv(ruta_datos + 'productos.csv')
df_sucursales_pd = pd.read_csv(ruta_datos + 'sucursales.csv')
df_clientes_pd = pd.read_csv(ruta_datos + 'clientes.csv')
df_ventas_pd = pd.read_csv(ruta_datos + 'ventas.csv', parse_dates=['fecha'])
df_detalles_pd = pd.read_csv(ruta_datos + 'detalles_ventas.csv')

print("✅ Datos cargados con Pandas")
print(f"  Productos: {len(df_productos_pd):,} registros")
print(f"  Ventas: {len(df_ventas_pd):,} registros")
print(f"  Detalles: {len(df_detalles_pd):,} registros")

In [0]:
# Convertir Pandas DataFrames a Spark DataFrames
df_productos = spark.createDataFrame(df_productos_pd)
df_sucursales = spark.createDataFrame(df_sucursales_pd)
df_clientes = spark.createDataFrame(df_clientes_pd)
df_ventas = spark.createDataFrame(df_ventas_pd)
df_detalles = spark.createDataFrame(df_detalles_pd)

print("✅ DataFrames convertidos a Spark")
print("\n📄 Schema de productos:")
df_productos.printSchema()

## Parte 2: Guardar como Tablas Delta

### 💾 Persistencia estructurada

Las tablas Delta permiten almacenar datos de forma eficiente con transacciones ACID, versionado y optimizaciones automáticas.

In [0]:
# Guardar DataFrames como tablas Delta en el catálogo
print("💾 Creando tablas Delta...\n")

# Productos
df_productos.write.mode('overwrite').saveAsTable('panaderia_productos')
print("✅ Tabla panaderia_productos creada")

# Sucursales
df_sucursales.write.mode('overwrite').saveAsTable('panaderia_sucursales')
print("✅ Tabla panaderia_sucursales creada")

# Clientes
df_clientes.write.mode('overwrite').saveAsTable('panaderia_clientes')
print("✅ Tabla panaderia_clientes creada")

# Ventas (particionada por año y mes para mejor rendimiento)
df_ventas.write.mode('overwrite') \
    .partitionBy('anio', 'mes') \
    .saveAsTable('panaderia_ventas')
print("✅ Tabla panaderia_ventas creada (particionada por anio/mes)")

# Detalles de ventas
df_detalles.write.mode('overwrite').saveAsTable('panaderia_detalles_ventas')
print("✅ Tabla panaderia_detalles_ventas creada")

print("\n✅ Todas las tablas Delta creadas exitosamente")

In [0]:
%sql
-- Ver las tablas creadas
SHOW TABLES LIKE 'panaderia_*'

## Parte 3: Agregaciones Complejas con Spark SQL

### 📈 Métricas de negocio agregadas

Vamos a crear tablas agregadas que resuman las ventas desde diferentes perspectivas.

In [0]:
# Leer las tablas recién creadas
ventas = spark.table('panaderia_ventas')
detalles = spark.table('panaderia_detalles_ventas')
productos = spark.table('panaderia_productos')
sucursales = spark.table('panaderia_sucursales')

# Crear tabla agregada: ventas por sucursal y mes
ventas_sucursal_mes = ventas.groupBy('sucursal_id', 'anio', 'mes').agg(
    F.count('venta_id').alias('cantidad_ventas'),
    F.sum('total').alias('facturacion_total'),
    F.avg('total').alias('ticket_promedio'),
    F.min('total').alias('venta_minima'),
    F.max('total').alias('venta_maxima')
).orderBy('anio', 'mes', 'sucursal_id')

# Guardar como tabla
ventas_sucursal_mes.write.mode('overwrite').saveAsTable('panaderia_ventas_sucursal_mes')

print("✅ Tabla de agregación creada: panaderia_ventas_sucursal_mes")
print("\n📄 Primeras filas:")
ventas_sucursal_mes.show(10)

In [0]:
# Unir detalles con productos para obtener nombres
ventas_productos = detalles.join(productos, 'producto_id', 'left')

# Agregar por producto
ventas_por_producto = ventas_productos.groupBy(
    'producto_id', 'nombre', 'categoria'
).agg(
    F.sum('cantidad').alias('unidades_vendidas'),
    F.sum('subtotal').alias('facturacion'),
    F.count('venta_id').alias('numero_ventas'),
    F.avg('precio_unitario').alias('precio_promedio')
).orderBy(F.desc('facturacion'))

# Guardar como tabla
ventas_por_producto.write.mode('overwrite').saveAsTable('panaderia_ventas_por_producto')

print("✅ Tabla de agregación creada: panaderia_ventas_por_producto")
print("\n🏆 Top 10 productos por facturación:")
ventas_por_producto.show(10, truncate=False)

In [0]:
%sql
-- Crear tabla agregada por categoría usando SQL puro
CREATE OR REPLACE TABLE panaderia_ventas_por_categoria AS
SELECT 
  p.categoria,
  COUNT(DISTINCT d.venta_id) as numero_ventas,
  SUM(d.cantidad) as unidades_vendidas,
  SUM(d.subtotal) as facturacion_total,
  AVG(d.precio_unitario) as precio_promedio,
  ROUND(AVG(d.descuento_porcentaje), 2) as descuento_promedio
FROM panaderia_detalles_ventas d
JOIN panaderia_productos p ON d.producto_id = p.producto_id
GROUP BY p.categoria
ORDER BY facturacion_total DESC;

SELECT * FROM panaderia_ventas_por_categoria;

## Parte 4: Funciones de Ventana (Window Functions)

### 🕹️ Cálculos por grupos con contexto

Las window functions permiten calcular rankings, acumulados y comparaciones dentro de particiones de datos.

In [0]:
# Calcular ranking de productos más vendidos por sucursal
from pyspark.sql.window import Window

# Unir ventas con detalles y productos
ventas_completas = ventas.join(detalles, 'venta_id').join(productos, 'producto_id')

# Agrupar por sucursal y producto
ventas_sucursal_producto = ventas_completas.groupBy(
    'sucursal_id', 'producto_id', 'nombre', 'categoria'
).agg(
    F.sum('cantidad').alias('unidades_vendidas'),
    F.sum('subtotal').alias('facturacion')
)

# Definir ventana: particionar por sucursal, ordenar por facturación
window_spec = Window.partitionBy('sucursal_id').orderBy(F.desc('facturacion'))

# Calcular ranking
ranking_productos = ventas_sucursal_producto.withColumn(
    'ranking', F.row_number().over(window_spec)
).filter(F.col('ranking') <= 5)  # Top 5 por sucursal

print("🏆 Top 5 productos por sucursal:")
ranking_productos.orderBy('sucursal_id', 'ranking').show(15, truncate=False)

In [0]:
# Calcular ventas acumuladas por mes
ventas_mensuales = ventas.groupBy('anio', 'mes').agg(
    F.sum('total').alias('facturacion_mensual')
).orderBy('anio', 'mes')

# Definir ventana para acumulado
window_acum = Window.orderBy('anio', 'mes').rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Calcular acumulado
ventas_acumuladas = ventas_mensuales.withColumn(
    'facturacion_acumulada', F.sum('facturacion_mensual').over(window_acum)
).withColumn(
    'crecimiento_vs_anterior', 
    F.round(
        (F.col('facturacion_mensual') - F.lag('facturacion_mensual').over(Window.orderBy('anio', 'mes'))) / 
        F.lag('facturacion_mensual').over(Window.orderBy('anio', 'mes')) * 100, 
        2
    )
)

print("📈 Ventas mensuales con acumulado y crecimiento:")
ventas_acumuladas.show(24)

## Parte 5: Ejercicios Prácticos

### ✍️ Ejercicios para Resolver

#### **Ejercicio 1**: Tabla agregada de clientes VIP
Crea una tabla que agregue las ventas de clientes VIP, mostrando cuánto ha gastado cada cliente VIP en total.

In [0]:
# EJERCICIO 1: Ventas de clientes VIP
# Pista: Une ventas con clientes, filtra por es_vip=True, agrupa y suma

# Tu código aquí:
clientes = spark.table('panaderia_clientes')

ventas_vip = ventas.join(
    clientes.filter(F.col('es_vip') == True),
    'cliente_id',
    'inner'
).groupBy('cliente_id', 'nombre').agg(
    F.count('venta_id').alias('numero_compras'),
    F.sum('total').alias('total_gastado'),
    F.avg('total').alias('ticket_promedio')
).orderBy(F.desc('total_gastado'))

print("🌟 CLIENTES VIP - TOP 10 POR GASTO")
ventas_vip.show(10, truncate=False)

#### **Ejercicio 2**: Participación de mercado por categoría
Calcula el porcentaje de facturación que representa cada categoría sobre el total.

In [0]:
%sql
-- EJERCICIO 2: Participación de mercado
-- Pista: Usa SUM() OVER() para calcular el total general

SELECT 
  p.categoria,
  SUM(d.subtotal) as facturacion,
  ROUND(SUM(d.subtotal) * 100.0 / SUM(SUM(d.subtotal)) OVER(), 2) as porcentaje_participacion
FROM panaderia_detalles_ventas d
JOIN panaderia_productos p ON d.producto_id = p.producto_id
GROUP BY p.categoria
ORDER BY facturacion DESC

## 🎯 Resumen del TP05

### ✅ Qué aprendimos:

1. **Tablas Delta**: Guardamos DataFrames como tablas persistentes con transacciones ACID
2. **Particionamiento**: Optimizamos consultas particionando por año/mes
3. **Agregaciones complejas**: Calculamos métricas agrupadas con groupBy y agg
4. **SQL en Spark**: Usamos tanto Python como SQL para crear tablas agregadas
5. **Window Functions**: Aplicamos rankings, acumulados y comparaciones con ventanas
6. **Transformaciones avanzadas**: Combinamos joins, agregaciones y funciones de ventana

### 📊 Tablas creadas:

* `panaderia_productos` - Catálogo de productos
* `panaderia_ventas` - Transacciones (particionada)
* `panaderia_ventas_sucursal_mes` - Agregado por sucursal y mes
* `panaderia_ventas_por_producto` - Agregado por producto
* `panaderia_ventas_por_categoria` - Agregado por categoría

### 🚀 Próximos pasos:

En el **TP06** aprenderemos a:
* Crear features para machine learning
* Aplicar encoding a variables categóricas
* Escalar y normalizar variables numéricas
* Preparar datasets para modelos predictivos

---

**📝 Excelente trabajo! Ahora sabes cómo estructurar y agregar datos a escala con Spark.**